# microGPT, on Wikipedia

Recreate Karpathy's microGPT, but trained on real Wikipedia text (`wikimedia/wikipedia`, Simple English) with our own byte-pair-encoding tokenizer and a GPT-2-style Transformer built from raw PyTorch tensor ops (`torch.autograd` for backward).

See `wiki/llm-from-scratch/microgpt.md` and `wiki/llm-from-scratch/nn-zero-to-hero.md`.

## [A] Setup & dependencies

Install deps with `uv` (`torch`, `datasets`), import them, and pick a device (CUDA / MPS / CPU) so the rest of the notebook stays device-agnostic.

In [ ]:
import math
import time

import torch

# Device-agnostic: prefer CUDA, then Apple MPS, else CPU.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# One seed for reproducible runs across data sampling, init, and batching.
torch.manual_seed(1337)

print(f"torch {torch.__version__} | device: {device}")

## [B] Config

All hyperparameters in one place: data subset size, BPE vocab size, `block_size`, `n_layer` / `n_head` / `n_embd`, learning rate, and step count.

In [ ]:
# --- Data ---
n_articles = 2000        # how many Wikipedia articles to pull into the corpus

# --- Tokenizer (BPE) ---
vocab_size = 1024        # final vocab: 256 raw bytes + (vocab_size - 256) learned merges

# --- Model ---
block_size = 128         # context length (tokens the model attends over)
n_layer = 4              # number of Transformer blocks
n_head = 4               # attention heads per block
n_embd = 128             # embedding / residual-stream width (must be divisible by n_head)
assert n_embd % n_head == 0, "n_embd must be divisible by n_head"

# --- Training ---
batch_size = 32          # sequences per step
learning_rate = 3e-4     # peak Adam learning rate
max_steps = 2000         # total optimizer steps
eval_interval = 200      # steps between val-loss checks
eval_iters = 50          # batches averaged per val-loss estimate

print(f"model: {n_layer}L / {n_head}H / {n_embd}D | block {block_size} | vocab {vocab_size}")

## [C] Load Wikipedia

Stream `wikimedia/wikipedia` config `20231101.simple` from the Hugging Face Hub and take the first N articles into a single raw-text corpus.

In [ ]:
import json
from pathlib import Path

from datasets import load_dataset

# Anchor a gitignored data/ dir at the repo root (walk up to find pyproject.toml).
_here = Path.cwd().resolve()
repo_root = next(p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists())
data_dir = repo_root / "data"
data_dir.mkdir(exist_ok=True)

# Single growing cache: one JSON-encoded article per line, in dataset order.
cache_path = data_dir / "corpus_simple.jsonl"
articles = []
if cache_path.exists():
    with cache_path.open(encoding="utf-8") as f:
        articles = [json.loads(line) for line in f]

have = len(articles)
if n_articles > have:
    # Need more than we have cached: .skip(have) prunes earlier parquet shards
    # so we start near the first absent record instead of decoding all of them.
    stream = load_dataset(
        "wikimedia/wikipedia", "20231101.simple", split="train", streaming=True
    )
    tail = stream.skip(have).take(n_articles - have)
    with cache_path.open("a", encoding="utf-8") as f:
        for row in tail:
            articles.append(row["text"])
            f.write(json.dumps(row["text"]) + "\n")
    print(f"appended {n_articles - have} articles (had {have}) -> {cache_path.name}")
else:
    print(f"using {n_articles} of {have} cached articles in {cache_path.name}")

# Take exactly n_articles; blank line between articles as a soft document boundary.
corpus = "\n\n".join(articles[:n_articles])
print(f"corpus: {len(corpus):,} chars")

## [D] Inspect the corpus

Sanity-check the data: total size, number of articles, and a sample article.

In [ ]:
used = articles[:n_articles]
lengths = [len(a) for a in used]

print(f"articles:        {len(used):,}")
print(f"total chars:     {len(corpus):,}  (~{len(corpus) / 1e6:.1f}M)")
print(f"total bytes:     {len(corpus.encode('utf-8')):,}")
print(f"chars/article:   min {min(lengths):,} | median {sorted(lengths)[len(lengths) // 2]:,} | max {max(lengths):,}")
print(f"distinct chars:  {len(set(corpus)):,}")

# Show one representative article (the median-length one), trimmed.
median_idx = sorted(range(len(used)), key=lambda i: lengths[i])[len(used) // 2]
sample = used[median_idx]
print(f"\n--- sample article (#{median_idx}, {len(sample):,} chars) ---")
print(sample[:800] + ("..." if len(sample) > 800 else ""))

## [E] Train a minimal BPE

Roll our own byte-pair-encoding: the merge loop that learns `vocab_size` merges from the corpus (the minbpe lesson, from scratch).

In [ ]:
# Byte-pair encoding from scratch: start from the 256 raw byte values and
# repeatedly merge the most frequent adjacent pair into a new token.
def get_stats(ids):
    """Count how often each adjacent token pair occurs."""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """Replace every occurrence of `pair` with a single new token `idx` (one L->R pass)."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


num_merges = vocab_size - 256        # the 256 byte values are already "tokens"
ids = list(corpus.encode("utf-8"))   # corpus as a flat stream of byte-tokens

merges = {}  # (int, int) -> int, insertion order == merge rank
t0 = time.time()
for k in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)  # most frequent adjacent pair
    idx = 256 + k
    ids = merge(ids, pair, idx)
    merges[pair] = idx
print(f"learned {len(merges)} merges in {time.time() - t0:.1f}s")

## [F] Encode / decode

Implement `encode` (apply merges) and `decode` (invert) and verify a roundtrip on sample text.

## [G] Tokenize the corpus

Encode the full corpus to token ids and split into train / val tensors.

## [H] Batching

`get_batch`: sample random contiguous (x, y) chunks of length `block_size` for next-token prediction.

## [I] Embeddings

Token + position embedding tables built from raw tensors.

## [J] RMSNorm

Root-mean-square normalization for training stability, from raw tensor ops.

## [K] Multi-head causal self-attention

The heart of the Transformer: Q/K/V projections, scaled dot-product attention with a causal mask, and output projection — all raw tensor math.

## [L] MLP block

Per-position feed-forward block (up-projection, nonlinearity, down-projection).

## [M] Transformer block

Compose one block: RMSNorm + attention + residual, then RMSNorm + MLP + residual.

## [N] GPT model

Assemble embeddings + a stack of blocks + final norm + LM head into a forward pass returning logits and cross-entropy loss.

## [O] Optimizer & LR schedule

Adam optimizer with a decaying learning-rate schedule.

## [P] Training loop

Run training steps with periodic validation-loss evaluation.

## [Q] Loss curves

Plot train / val loss over steps to see learning progress.

## [R] Sampling

Autoregressive generation: sample the next token from the model's distribution with a temperature knob.

## [S] Generate

Produce Wikipedia-style text samples from the trained model.

## [T] Save / load

Persist the BPE merges and model `state_dict` to `artifacts/` (gitignored) so a trained run can be reloaded without retraining.